# AoA Estimation

## Imports

In [1]:
from dataset_RT import training_set
import tensorflow as tf
import numpy as np
import tqdm

2026-02-09 12:57:52.637217: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770641872.659281    6372 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770641872.666038    6372 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770641872.683882    6372 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770641872.683900    6372 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770641872.683902    6372 computation_placer.cc:177] computation placer alr

## Extract Specs of the CSI Data

In [2]:
# Count number of datapoints in training set (for progress bar)
TOTAL_DATAPOINTS = sum(1 for _ in training_set)
print("Total number of datapoints: {}".format(TOTAL_DATAPOINTS))

# Determine antenna array dimensions from the first datapoint
# CSI shape: (arrays, rows, columns, subcarriers, 2)
ANTENNAS_PER_ROW = tf.shape(training_set.take(1).get_single_element()[0])[2].numpy()
ANTENNAS_PER_COLUMN = tf.shape(training_set.take(1).get_single_element()[0])[1].numpy()
print("Number of antennas per row: {}".format(ANTENNAS_PER_ROW))
print("Number of antennas per column: {}".format(ANTENNAS_PER_COLUMN))

2026-02-09 12:57:56.157416: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:381] TFRecordDataset `buffer_size` is unspecified, default to 262144


Total number of datapoints: 5000
Number of antennas per row: 8
Number of antennas per column: 8


2026-02-09 12:57:57.048195: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## Estimate AoAs

In [3]:
def get_unitary_rootmusic_estimator(chunksize = 32, shed_coeff_ratio = 0):
    I = np.eye(chunksize // 2)
    J = np.flip(np.eye(chunksize // 2), axis = -1)
    Q = np.asmatrix(np.block([[I, 1.0j * I], [J, -1.0j * J]]) / np.sqrt(2))
    
    def unitary_rootmusic(R):
        assert(len(R) == chunksize)
        C = np.real(Q.H @ R @ Q)
    
        eig_val, eig_vec = np.linalg.eigh(C)
        eig_val = eig_val[::-1]
        eig_vec = eig_vec[:,::-1]

        source_count = 1
        En = eig_vec[:,source_count:]
        ENSQ = Q @ En @ En.T @ Q.H
    
        coeffs = np.asarray([np.trace(ENSQ, offset = diag) for diag in range(1, len(R))])
        coeffs = coeffs[:int(len(coeffs) * (1 - shed_coeff_ratio))]

        # Remove some of the smaller noise coefficients, trade accuracy for speed
        coeffs = np.hstack((coeffs[::-1], np.trace(ENSQ), coeffs.conj()))
        roots = np.roots(coeffs)
        #roots = roots[abs(roots) < 1.0]
        largest_root = np.argmax(1 / (1.0 - np.abs(roots)))
        
        return np.angle(roots[largest_root])

    return unitary_rootmusic

In [4]:
umusic_azimuth = get_unitary_rootmusic_estimator(ANTENNAS_PER_ROW)
umusic_elevation = get_unitary_rootmusic_estimator(ANTENNAS_PER_COLUMN)

estimated_aoas_azimuth = []
estimated_aoas_elevation = []

for index, data in enumerate(tqdm.tqdm(training_set, total = TOTAL_DATAPOINTS)):
    csi, pos = data[0], data[1]

    # Estimate AoAs based on all subcarriers
    
    # Compute covariance matrix for azimuth estimation (across antenna columns for each array)
    R_azimuth = np.einsum("arms,arns->amn", csi, np.conj(csi))  # Shape: (arrays, columns, columns)
    # Compute covariance matrix for elevation estimation (across antenna rows for each array)
    R_elevation = np.einsum("arms,aqms->arq", csi, np.conj(csi))  # Shape: (arrays, rows, rows)

    # Apply root-MUSIC to estimate azimuth AoA for each receiver array
    estimated_aoas_azimuth.append([np.arcsin(umusic_azimuth(R_azimuth[array]) / np.pi) for array in range(len(R_azimuth))])
    # Apply root-MUSIC to estimate elevation AoA for each receiver array
    estimated_aoas_elevation.append([np.arcsin(umusic_elevation(R_elevation[array]) / np.pi) for array in range(len(R_elevation))])

100%|██████████| 5000/5000 [00:28<00:00, 175.92it/s]


## Save AoAs as NumPy

In [5]:
# Save AoAs as NumPy
estimated_aoas_azimuth = np.asarray(estimated_aoas_azimuth)
estimated_aoas_elevation = np.asarray(estimated_aoas_elevation)

np.save("results/estimated_aoas_azimuth.npy", estimated_aoas_azimuth)
np.save("results/estimated_aoas_elevation.npy", estimated_aoas_elevation)